# COMP0005 - GROUP COURSEWORK
# Experimental Evaluation of Search Data Structures and Algorithms

The cell below defines **AbstractSearchInterface**, an interface to support basic insert/search operations; you will need to implement this three times, to realise your three search data structures of choice among: (1) *2-3 Tree*, (2) *AVL Tree*, (3) *LLRB BST*; (4) *B-Tree*; and (5) *Scapegoat Tree*. <br><br>**Do NOT modify the next cell** - use the dedicated cells further below for your implementation instead. <br>

In [ ]:
# DO NOT MODIFY THIS CELL

from abc import ABC, abstractmethod  

class AbstractSearchInterface(ABC):
    '''
    Abstract class to support search/insert operations (plus underlying data structure)
    
    '''
        
    @abstractmethod
    def insertElement(self, element):     
        '''
        Insert an element in a search tree
            Parameters:
                    element: string to be inserted in the search tree (string)

            Returns:
                    "True" after successful insertion, "False" if element is already present (bool)
        '''
        
        pass 
    

    @abstractmethod
    def searchElement(self, element):
        '''
        Search for an element in a search tree
            Parameters:
                    element: string to be searched in the search tree (string)

            Returns:
                    "True" if element is found, "False" otherwise (bool)
        '''

        pass

Use the cell below to define any auxiliary data structure and python function you may need. Leave the implementation of the main API to the next code cells instead.

In [ ]:
# ADD AUXILIARY DATA STRUCTURE DEFINITIONS AND HELPER CODE HERE

class AVLNode:
    def __init__(self, key: str):
        self.key: str = key
        self.height: int = 0
        self.left: AVLNode | None = None
        self.right: AVLNode | None = None


class LLRBNode:
    def __init__(self, key):
        self.key: str = key
        self.left: LLRBNode | None = None
        self.right: LLRBNode | None = None
        self.color: bool = True # True: red, False: black

class LLRBHelper:

    @staticmethod
    def rotate_left(root: LLRBNode) -> LLRBNode:
        right = root.right
        root.right = right.left
        right.left = root

        right.color = root.color
        root.color = True

        return right

    @staticmethod
    def rotate_right(root: LLRBNode) -> LLRBNode:
        left = root.left
        root.left = left.right
        left.right = root

        left.color = root.color
        root.color = True

        return left

    @staticmethod
    def flip_colors(node: LLRBNode) -> LLRBNode:
        node.color = not node.color
        node.left.color = not node.left.color
        node.right.color = not node.right.color
        return node

    @staticmethod
    def is_red(node: LLRBNode | None) -> bool:
        return node is not None and node.color

    @staticmethod
    def search(key: str, root: LLRBNode | None) -> bool:
        while root:
            if key < root.key:
                root = root.left
            elif key > root.key:
                root = root.right
            else:
                return True
        return False

    @staticmethod
    def insert(key: str, root: LLRBNode | None) -> LLRBNode:
        if not root:
            return LLRBNode(key)
        if key < root.key:
            root.left = LLRBHelper.insert(key, root.left)
        elif key > root.key:
            root.right = LLRBHelper.insert(key, root.right)
        else:
            return root
        if LLRBHelper.is_red(root.right) and not LLRBHelper.is_red(root.left):
            root = LLRBHelper.rotate_left(root)
        if LLRBHelper.is_red(root.left) and LLRBHelper.is_red(root.left.left):
            root = LLRBHelper.rotate_right(root)
        if LLRBHelper.is_red(root.left) and LLRBHelper.is_red(root.right):
            root = LLRBHelper.flip_colors(root)

        return root

class TwoThreeNode:
  '''A node in a 2-3 tree.

  Attributes:
    keys: A sorted list of one or two keys (can have three keys but only temporarily)
    children: A list of zero (leaf), two (2-node) or three (3-node) child nodes
  '''
  def __init__(self, keys=None, children=None):
    self.keys = keys or []
    self.children = children or []
  def is_leaf(self):
    return len(self.children) == 0

Use the cell below to implement the requested API by means of **2-3 Tree** (if among your chosen data structure).

In [ ]:
'''
2-3 Tree Implementation:

A 2-3 tree is a balanced search tree in which
- 2 nodes have one key and two children
- 3 nodes have two keys and three children
- All the leaf nodes lie at the same depth

The tree is balanced by splitting 'overfull' nodes (temporary nodes with 3 keys and 4 children) into two 2 nodes, and passing the middle key up the tree. The tree depth can only be increased by a split operation at the root node: this is how the tree maintains balance.
Splitting is a constant time operation and only acts locally in the structure.

Searching a 2-3 tree of N nodes is an O(logN) operation; inserting a node into a 2-3 tree of size N is also an O(logN) operation. The depth of the tree ranges from log N / log 2 (log base 2 of N), to log N / log 3 (log base 3 of N).
'''

class TwoThreeTree(AbstractSearchInterface):
        
  '''A 2-3 balanced search tree with 'search' and 'insert' operations.
  '''
  def __init__(self, root=None):
    self.root = root

  def searchElement(self, element):
    '''Returns true if and only if the 2-3 tree contains a specified element. Delegates work to recursive helper, 'search'.

    Args:
        node (Node): the root of the tree/subtree to search
        element (_): the element to search for
    '''
    if self.root is None:
      return False
    return self.__search(self.root, element)
  
  def __search(self, node, element):
    '''Returns true if and only if the subtree rooted at a given node contains a specified element.

    Args:
        node (Node): the root of the tree/subtree to search
        element (_): the element to search for
    
    '''

    if node is None:
      return False
    
    # Checks if the element is in this node
    if element in node.keys:
      return True
    
    # Leaf reached without finding the element -> the element is not in the tree
    if node.is_leaf():
      return False
    
    # Pick the correct child to descend the tree to
    # The index of the first key greater than the element is the index of the correct child
    # (If no key is greater, then descend to the rightmost child)
    child_index = len(node.keys)
    for i, key in enumerate(node.keys):
      if element < key:
        child_index = i
        break

    return self.__search(node.children[child_index], element)
  
  def insertElement(self, element):
    '''Insert a element into a 2-3 tree, keeping the balance in check. Delegates work to the recursive _insert method.

    Args:
        element (_): the element to insert
    Returns:
        True if and only if the element is successfully inserted into the tree; False otherwise (if there are duplicates)
    '''
    # If the tree is empty, then create a single leaf root
    if self.root is None:
      self.root = TwoThreeNode([element])
      return True
    
    result = self.__insertElement(self.root, element)
    
    # Result is false -> element already in the tree -> return false
    if result is False:
      return False
    
    # If the root split, then create a new one, one level higher
    # This is the only way the tree can grow taller
    if result is not None:
      middle_key, left_child, right_child = result
      self.root = TwoThreeNode([middle_key], [left_child, right_child])
    
    return True 
  
  def __insertElement(self, node, element):
    '''Insert a element into a 2-3 subtree rooted at a given node, keeping the balance in check. This is a recurisve helper method for insert.

    Args:
        node (Node): the root node of the subtree
        element (_): the element to insert

    Returns:
        None : if the insertion is absorbed by the node without causing a split operation
        middle_key, left_child, right_child (_, Node, Node) : if the node split, the calling method must absorb the key being passed up, and the left and right children
        False : if the element being inserted is already in the tree
    '''
    # Element already in tree -> return false
    if element in node.keys:
      return False
    
    # Base case - leaf node
    if node.is_leaf():

      node.keys.append(element)
      node.keys.sort()

      if len(node.keys) <= 2: # Still a valid 2- or 3- node
        return None
      
      # Overfull (has 3 keys) -> split and pass the middle key up the call chain
      middle_key = node.keys[1]
      left_child = TwoThreeNode([node.keys[0]])
      right_child = TwoThreeNode([node.keys[2]])
      return (middle_key, left_child, right_child)
          
    # Recursive case - 2- or 3- (non-leaf) node
    # Find the right child to recurse to (same logic as in search)
    child_index = len(node.keys)
    for i, key in enumerate(node.keys):
      if element < key:
        child_index = i
        break

    result = self.__insertElement(node.children[child_index], element)
    
    # result False -> duplicate data -> propagate this up the call stack 
    if result is False:
      return False
    if result is None: # Child absorbed the insert without splitting - so do nothing
      return None   
    
    # A child split, so absorb the recieved key and new children
    middle_key, left_child, right_child = result
    node.keys.append(middle_key)
    node.keys.sort()
    # Replace the child at child_index with the two new children
    node.children = node.children[:child_index] + [left_child, right_child] + node.children[child_index+1:]

    if len(node.keys) <= 2: # Still a valid 2- or 3- node - so do not initiate a split
      return None
    # Overfull (has 3 keys) -> split and pass (as before) but attach children
    # left_child node gets the first two children, right_child node get the last two
    middle_key = node.keys[1] 
    left_child = TwoThreeNode([node.keys[0]], node.children[:2])
    right_child = TwoThreeNode([node.keys[2]], node.children[2:])
    return (middle_key, left_child, right_child)

In [ ]:
'''Extremely basic test (temporary)
'''

from random import shuffle 
bst = TwoThreeTree()


shuffled_even_integers = [i for i in range(0, 100, 2)]
shuffle(shuffled_even_integers)

for i in shuffled_even_integers:
  bst.insertElement(i)

positives = [] # Expect all even integers 0 - 100
negatives = [] # Expect all odd integers 0 - 100
integer_range = [i for i in range(100)]
for i in integer_range:
  if bst.searchElement(i):
    positives += [i]
  else:
    negatives += [i]

print("The elements in the tree (should be even integers 0-100) are: ", positives)
print("The elements not in the tree (should be odd integers 0-100) are: ", negatives)

# Note: I wrote this to verify my implementation works - we need to agree on a rigorous testing framework

Use the cell below to implement the requested API by means of **AVL Tree** (if among your chosen data structure).

In [ ]:
class AVLTree(AbstractSearchInterface):

    @staticmethod
    def heightOf(root: AVLNode | None) -> int:
        # height of root = edges from root to furthest leaf
        # -1 because height(leaf) = 1 + max(-1,-1) = 0
        if root is None:
            return -1
        else:
            return 1 + max(
                root.left.height if root.left is not None else -1,
                root.right.height if root.right is not None else -1
            )


    @staticmethod
    def rotateRight(root: AVLNode) -> AVLNode:
        # returns the new root of the subtree (currently root.left)
        assert root.left is not None, "Right rotation requires root.left to be a Node"
        
        # perform rotation
        left = root.left
        leftRight = root.left.right
        root.left = leftRight
        left.right = root

        #update heights
        root.height = AVLTree.heightOf(root)
        left.height = AVLTree.heightOf(left)

        return left


    @staticmethod
    def rotateLeft(root: AVLNode) -> AVLNode:
        # returns the new root of the subtree (root.right)
        assert root.right is not None, "Left rotation requires root.right to be a Node"

        # perform rotation
        right = root.right
        rightLeft = root.right.left
        root.right = rightLeft
        right.left = root

        # update heights
        root.height = AVLTree.heightOf(root)
        right.height = AVLTree.heightOf(right)

        return right   
    

    @staticmethod
    def searchHelper(element: str, root: AVLNode | None) -> bool:
        # returns whether the element is in root's subtree
        if root == None:
            return False
        elif root.key == element:
            return True
        elif root.key > element:
            return AVLTree.searchHelper(element, root.left)
        else:
            return AVLTree.searchHelper(element, root.right)
    

    @staticmethod
    def insertHelper(element: str, root: AVLNode | None) -> AVLNode:
        # insert into the right/left subtree
        if root is None:
            return AVLNode(element)
        elif root.key > element:
            root.left = AVLTree.insertHelper(element, root.left)
        elif root.key < element:
            root.right = AVLTree.insertHelper(element, root.right)

        # root.left and root.right are now balanced
        # height of child nodes may have changed, so height of root may have changed
        root.height = AVLTree.heightOf(root)        

        # balance of root = left height - right height
        # valid AVL tree has valid balance values of -1,0,1
        balance = AVLTree.heightOf(root.left) - AVLTree.heightOf(root.right)

        # case 1: left subtree too heavy and imbalance from left's left subtree
        if balance > 1 and root.left is not None and element < root.left.key:
            return AVLTree.rotateRight(root)
        # case 2: left subtree too heavy and imbalance from left's right subtree
        elif balance > 1 and root.left is not None and element > root.left.key:
            root.left = AVLTree.rotateLeft(root.left)
            return AVLTree.rotateRight(root)
        # case 3: mirror case 1
        if balance < -1 and root.right is not None and element > root.right.key:
            return AVLTree.rotateLeft(root)
        # case 4: mirror case 2
        elif balance < -1 and root.right is not None and element < root.right.key:
            root.right = AVLTree.rotateRight(root.right)
            return AVLTree.rotateLeft(root)
        # else balanced
        else:
            return root
    

    @staticmethod
    def printBfs(root: AVLNode | None) -> None:
        q: list[AVLNode | None] = [root]
        while len(q) > 0:
            node = q.pop(0)
            if node:
                print(node.key)
                q.append(node.left)
                q.append(node.right)


    def __init__(self, root=None):
        self.root = root

    
    def insertElement(self, element: str) -> bool:
        canInsert = not AVLTree.searchHelper(element, self.root)
        if canInsert:
            self.root = AVLTree.insertHelper(element, self.root)
        return canInsert
    
    
    def searchElement(self, element: str) -> bool:             
        found = AVLTree.searchHelper(element, self.root)
        return found

In [ ]:
# tmp test block

r = AVLNode("a")
t = AVLTree(r)
for i in range(98, 98+14):
  t.insertElement(chr(i))
AVLTree.printBfs(t.root)

Use the cell below to implement the requested API by means of **LLRB BST** (if among your chosen data structure).

In [ ]:
class LLRBBST(AbstractSearchInterface):

    def __init__(self, root=None):
        self.root = root
        
    def insertElement(self, element):
        inserted = False
        # ADD YOUR CODE HERE
        if self.searchElement(element):
            return False
        self.root = LLRBHelper.insert(element, self.root)
        self.root.color = False
        inserted = True
        return inserted
    
    

    def searchElement(self, element):     
        found = False
        # ADD YOUR CODE HERE
        found = LLRBHelper.search(element, self.root)
        
        return found

In [ ]:
# Test
def print_tree(node, indent="", side="root"):
    if node is None:
        print(f"{indent}[{side}] None")
        return

    color = "R" if node.color else "B"
    print(f"{indent}[{side}] {node.key}({color})")
    print_tree(node.left, indent + "   ", "L")
    print_tree(node.right, indent + "   ", "R")

tree = LLRBBST()

print(tree.insertElement("10"))
print(tree.insertElement("20"))
print(tree.insertElement("30"))
print(tree.insertElement("20"))

print(tree.searchElement("10"))
print(tree.searchElement("20"))
print(tree.searchElement("30"))
print(tree.searchElement("40"))

print("root color is black:", tree.root.color == False)

print_tree(tree.root)

Use the cell below to implement the requested API by means of **B-Tree** (if among your chosen data structure).

In [ ]:
class BTree(AbstractSearchInterface):
        
    def insertElement(self, element):
        inserted = False
        # ADD YOUR CODE HERE
      
        
        return inserted
    
    

    def searchElement(self, element):     
        found = False
        # ADD YOUR CODE HERE

        
        return found

Use the cell below to implement the requested API by means of **Scapegoat Tree** (if among your chosen data structure).

In [ ]:
class ScapegoatTree(AbstractSearchInterface):
        
    def insertElement(self, element):
        inserted = False
        # ADD YOUR CODE HERE
      
        
        return inserted
    
    

    def searchElement(self, element):     
        found = False
        # ADD YOUR CODE HERE

        
        return found 

Use the cell below to implement the **synthetic data generator** needed by your experimental framework (be mindful of code readability and reusability).

In [ ]:
import string
import random

class TestDataGenerator():
    '''
    A class to represent a synthetic data generator.

    ...

    Attributes
    ----------
    
    [to be defined as part of the coursework]

    Methods
    -------
    
    [to be defined as part of the coursework]

    '''
    
    #ADD YOUR CODE HERE
    
    def __init__(self, str_len: int):
        self.__str_len: int = str_len
        self.__lowerBound: int = 0
        self.__upperBound: int = 10 ** str_len
        self.__normal_mean: float = (self.__lowerBound + self.__upperBound) / 2
        self.__normal_sd: float = (self.__upperBound - self.__normal_mean) / 3 
        

    '''
    Cast the value to int (truncate if it is a float)
    Cast the value to a string, and left-pad 0's to make all keys equal size in memory
    Left-pad does not change lexicographic order
    '''
    def __format(self, value: int | float):
        return str(int(value)).zfill(self.__str_len)


    def next_random_uniform_value(self):
        while True:
            value = random.randint(self.__lowerBound, self.__upperBound)
            yield self.__format(value)
    

    ''' 
    Draws a random value from (a slightly truncated) normal distribution
    99.7% of values are within the lower and upper bound on keys
    Keys outside are discarded and rerolled
    '''
    def next_random_normal_value(self):
        while True:
            value = random.gauss(self.__normal_mean, self.__normal_sd)
            if self.__lowerBound <= value <= self.__upperBound:
                yield self.__format(value)


    def next_sorted_value(self):
        c = 0
        while True:
            yield self.__format(c)
            c = (c + 1) % self.__upperBound
    

    def next_reverse_sorted_value(self):
        c = self.__upperBound
        while True:
            yield self.__format(c)
            c = (c - 1) if c > 0 else self.__upperBound


    def next_constant_value(self):
        while True:
            yield self.__format(self.__normal_mean)

Use the cell below to implement the requested **experimental framework** (be mindful of code readability and reusability).

In [ ]:
import timeit
import matplotlib

class ExperimentalFramework():
    '''
    A class to represent an experimental framework.

    ...

    Attributes
    ----------
    
    [to be defined as part of the coursework]

    Methods
    -------
    
    [to be defined as part of the coursework]

    '''
            
    #ADD YOUR CODE HERE
    
    def __init__(self):
        pass

    
    ''' 
    dataGenerator is a generator from the TestDataGenerator class
    '''
    def build_structure(self, n, implementation: type[AbstractSearchInterface], dataGenerator):
        structure = implementation()
        for _ in range(n):
            key = next(dataGenerator)
            structure.insertElement(key)
        return structure
    
    
    def time_operation(self, operationMethod, key):
        return timeit.timeit(lambda: operationMethod(key), number=1)


    '''
    Times m insertions on a structure (subclass of AbstractSearchInterface) of size n
    Reconstructs the tree after every insert due to no deletion operation
        Change of size between tests would invalidate the fact that it is size n
    '''
    def time_m_inserts_on_structure_size_n(self, implementation: type[AbstractSearchInterface], dataGeneratorMethod, n, m):
        times = []
        for _ in range(m):
                dataGenerator = dataGeneratorMethod()
                structure = self.build_structure(n, implementation, dataGenerator)
                operationMethod = getattr(structure, 'insertElement')
                key = "" # need to decide this -> how do we choose what key to time insertions on? random?
                time = self.time_operation(operationMethod, key)
                times.append(time)
        return times


    ''' 
    Does not reconstruct tree after every search as the size has not changed
    '''
    def time_m_searches_on_structure_size_n(self, implementation: type[AbstractSearchInterface], dataGeneratorMethod, n, m):
        times = []
        dataGenerator = dataGeneratorMethod()
        structure = self.build_structure(n, implementation, dataGenerator)
        operationMethod = getattr(structure, 'insertElement')
        for _ in range(m):
                key = "" # need to decide this -> how do we choose what key to time insertions on? random?
                time = self.time_operation(operationMethod, key)
                times.append(time)
        return times


    def time_implementations(self, key_str_len):
        dataGenerator = TestDataGenerator(key_str_len)
        implementations = (
            TwoThreeTree,
            LLRBBST,
            AVLTree
        )
        operations = (
            self.time_m_inserts_on_structure_size_n, 
            self.time_m_searches_on_structure_size_n
        )
        dataGeneratorMethods = (
            dataGenerator.next_random_uniform_value,
            dataGenerator.next_random_normal_value,
            dataGenerator.next_sorted_value,
            dataGenerator.next_reverse_sorted_value,
            dataGenerator.next_constant_value
        )
        ns = [1, 10, 10**2, 5*10**2, 10**3, 5*10**3, 10**4]
        m = 5
        implementationResults = dict()
        for implementation in implementations:
            operationsResults = dict()
            for operation in operations:
                generatorsResults = dict()
                for dataGeneratorMethod in dataGeneratorMethods:
                    timeResults = dict()
                    for n in ns:
                        times = operation(implementation, dataGeneratorMethod, n, m)
                        timeResults[n] = times
                    generatorsResults[dataGeneratorMethod.__name__] = timeResults
                operationsResults[operation.__name__] = generatorsResults
            implementationResults[implementation.__name__] = operationsResults
        return implementationResults



Use the cell below to illustrate the python code you used to **fully evaluate** your three chosen search data structures and algortihms. The code below should illustrate, for example, how you made used of the **TestDataGenerator** class to generate test data of various size and properties; how you instatiated the **ExperimentalFramework** class to  evaluate each data structure using such data, collect information about their execution time, plot results, etc. Any results you illustrate in the companion PDF report should have been generated using the code below.

In [ ]:
import math

class StatisticsFramework:
  ''' Calculates some statistics / fits a regression line to the data.

  - mean, variance, standard_deviation and standard_error methods operate on a list of input numerical data.
  - The display_basic_statistics method shows you how to use them properly (and the correct units)

  - logarithmic fit returns the coefficients that best fit y = a + b log(x)
  - get_logarithmic_PMCC returns the PMCC for this model
  - get_logarithmic_r2 returns the coefficient of determination (the PMCC squared)

  - get_logarithmic_residuals returns a list of residuals (actual - predicted) for the model.
  It takes in x_data, y_data, a, b where a and b are the coefficients.
  It would be useful to plot this against log(x) using a scatter graph - since it visualises the goodness-of-fit of our model.
  We would expect randomly distributed data points about 0. 

  - If we can't do this plot, then get_RMSE takes in the list of residuals and calculates their variation about 0. 
  This is another decent (but not as good) way of showing the goodness-of-fit.
  - See display_logarithmic_fit_statistics for an example of use.

  The whole motivation to evaluate the goodness-of-fit of a logarithmic model is that it will show our trees are operating in-line with theory.
  (We expect insert and search to be O(logN) time algorithms) for all trees.

  - (IMPORTANT) Ideally, the standard errors for each sample should be plotted as vertical error bars on our graphs.
  They show the region in which the sample means are likely to fall into. These work (approximately) like standard deviations -
  we expect ~68% of sample means to fall within 1 std-err of our data point. ~95% within 2 std-errors, and 99.7% within 3. 
  The error bars will give an evaluation of how confident we are in our measurements. 

  - Also, I've kept the methods pretty modular, and so there are some unnecessarily repeated calculations.
  But I've kept it like this because (I'm hoping) it makes the class easier to use.
  Talk to me if you're stuck. 
  '''
        
  def mean(self, data):
    return sum(data) / len(data)
  
  def variance(self, data):
    ''' Calculates the variance in a list of numerical data.
    '''
    n = len(data)
    mean = self.mean(data)
    Sxx = self.__sum_squared_deviation(data, mean)
    return Sxx / (n-1) # this is the sample variance
  
  def standard_deviation(self, data):
    '''Calculates the standard deviation in a list of numerical data.
    '''
    return math.sqrt(self.variance(data))
    
  def standard_error(self, data):
    '''Returns the standard error of the mean for a list of numerical data.
    Equal to, the sample standard deviation / sqrt(n), where n is the number of elements in the list.
    The standard error estimates the standard deviation of the of distribution of sample means. That is, an estimate of how the sample mean varies over repeated samples. 
    '''
    n = len(data)
    return self.standard_deviation(data) / math.sqrt(n)
    
  def logarithmic_fit(self, x_data, y_data):
    '''Return the coefficients for the logarithmic regression of y_data on x_data.
    That is, the values (a,b) such that y = a + b * ln(x) best fits the data.
    '''

    log_x_data = self.__natural_log_data(x_data)
    (a,b) = self.__linear_fit(log_x_data, y_data)

    return (a, b)
  
  def get_logarithmic_PMCC(self, x_data, y_data):
    '''Returns the PMCC of the regression line of y_data on log(x_data).
    '''
    log_x_data = self.__natural_log_data(x_data)
    return self.__get_linear_PMCC(log_x_data, y_data)
  
  def get_logarithmic_r2(self, x_data, y_data):
    ''' Returns r-squared (the PMCC squared). 
    This is a value in the interval [0,1] that tells us the proportion of the variance in the data that the fitted model accurately captures.
    '''
    return self.get_logarithmic_PMCC(x_data, y_data) ** 2
  
  def get_logarithmic_residuals(self, x_data, y_data, a, b):
    '''Return the residuals (actual - predicted) values for the logarithmic fit as a list
    Parameters 'a' and 'b' are the values a, b in the model y = a + b * log(x)
    '''
    n = len(x_data)

    f = lambda x: a + b * math.log(x)

    return [y_data[i] - f(x_data[i]) for i in range(n)]
  
  def get_RMSE(self, residuals):
    '''Returns the RMSE of the model for the fitted data. This is the Root of the Mean of the Squared Errors.
    It is a measure of the spread (standard deviation) of the errors about 0 
    Given by: (the sum of the squares of the residuals / n)^1/2
    '''
    n = len(residuals)
    return math.sqrt(sum([r**2 for r in residuals]) / n)
  
  def display_basic_statistics(self, data):
    '''Calculates and prints the following statistics about a list of numeric data:
    sample mean, sample variance, sample standard deviation and sample standard error
    '''

    sample_mean = self.mean(data)
    sample_variance = self.variance(data)
    sample_std_dev = self.standard_deviation(data)
    sample_std_err = self.standard_error(data)

    # Note: I'm not sure about the units of measurement we are using for time. I have assumed seconds for now.
    print("The basic statistics are: ")
    print(f"The sample mean is: {sample_mean} seconds")
    print(f"The sample variance is: {sample_variance} seconds^2")
    print(f"The sample standard deviation is: {sample_std_dev} seconds")
    print(f"The sample standard error is: {sample_std_err} seconds")
    print(f"\n")

  def display_logarithmic_fit_statistics(self, x_data, y_data):
    '''Fits a logarithmic model, y = a + b * log(x), to y_data and x_data. 
    And gives the following statistics: 
    coefficents of the line of best fit, the PMCC for that line, r^2 (also known as the coefficient of determination, and the RMSE)
    Note, the class also has a method to provide the residuals as list; but this data needs to be plotted.
    '''

    a, b = self.logarithmic_fit(x_data, y_data)
    pmcc = self.get_logarithmic_PMCC(x_data, y_data)
    r2 = self.get_logarithmic_r2(x_data, y_data)
    residuals = self.get_logarithmic_residuals(x_data, y_data, a, b)
    rmse = self.get_RMSE(residuals)

    # Note: I'm not sure about the units of measurement we are using for time. I have assumed seconds for now.
    print("The regression statistics are: ")
    print(f"The model of best fit is: y = {a} + {b}*log(x)")
    print(f"The PMCC is: {pmcc}")
    print(f"The coefficient of determination (r^2) is: {r2}")
    print(f"The RMSE is: {rmse} seconds")
    print(f"\n")

  def __natural_log_data(self, data):
    '''Return a list of the natural logarithms of each value.
    '''
    if any([x <= 0 for x in data]):
      raise ValueError("You cannot log a non-positive number - check logic.")
    return [math.log(i) for i in data]
  
  def __linear_fit(self, x_data, y_data):
    '''Return the coefficients for the linear regression of y_data on x_data.
    Being, (a, b) where y = a + b * x best fits the data.
    Using the formulae: b = Sxy / Sxx - the ratio of covariance and variance
    And, a = y_mean - b * x_mean - the y-intercept of a line with gradient b passing through (x_mean, y_mean)
    '''
    x_mean, y_mean, Sxx, _, Sxy = self.__get_summary_statistics(x_data, y_data)

    b = Sxy / Sxx # Equivalent to the covariance / variance
    a = y_mean - b * x_mean # Line of best fit passes through (x_mean, y_mean)

    return (a, b)
  
  def __get_linear_PMCC(self, x_data, y_data):
    '''Return the product moment correlation coefficient of the linear relationship between y_data and x_data
    Calculation is equivalent to co-variance of x_data and y_data / (standard deviation in x_data * standard deviation in y_data)
    '''

    _, _, Sxx, Syy, Sxy = self.__get_summary_statistics(x_data, y_data)
    return Sxy / math.sqrt(Sxx * Syy)
  
  def __get_summary_statistics(self, x_data, y_data):
    '''Returns some summative statistics about the data used frequently in regression calculations.
    '''

    x_mean = self.mean(x_data)
    y_mean = self.mean(y_data)

    Sxx = self.__sum_squared_deviation(x_data, x_mean)
    Syy = self.__sum_squared_deviation(y_data, y_mean)
    Sxy = self.__sum_co_deviations(x_data, y_data, x_mean, y_mean)

    return (x_mean, y_mean, Sxx, Syy, Sxy)
 
  def __sum_co_deviations(self, x_data, y_data, x_mean, y_mean):
    '''Returns S_xy (a summary statistic for n * covariance) between two lists of numerical data.
    '''
    if len(x_data) != len(y_data):
      raise ValueError("x_data and y_data points are mismatched")
    n = len(x_data)

    return sum([(x_data[i] - x_mean) * (y_data[i] - y_mean) for i in range(n)])
  
  def __sum_squared_deviation(self, x_data, x_mean):
    ''' Returns S_xx (a summary statistic for n * variance) in a list of numerical data.
    '''
    n = len(x_data)

    return sum([(x_data[i] - x_mean)**2 for i in range(n)])

In [ ]:
'''This is a basic demo / test of the stats framework. 
'''
import random

sf = StatisticsFramework()
a, b = 37, 42 # Random test numbers
max_random_offset = 2
x_data = [i for i in range(1, 10)]
f = lambda x: a + b * math.log(x)

y_data = [f(x) for x in x_data]
# Generate y values that closely follow y = a + b log x relationship with some random offset
y_random_offset_data = [y + random.uniform(-max_random_offset, max_random_offset) for y in y_data]

sf.display_basic_statistics(x_data)

sf.display_logarithmic_fit_statistics(x_data, y_data)
sf.display_logarithmic_fit_statistics(x_data, y_random_offset_data)

In [ ]:
# ADD YOUR TEST CODE HERE 
import json

e = ExperimentalFramework()
data = e.time_implementations(7)
print(json.dumps(data))

